# 01 · Basics: one call, streaming, async batches

Tutorial sections 1–3 ([`TUTORIAL.md`](../TUTORIAL.md)).

All models are reached through AWS Bedrock. Run the setup cell first: it loads `.env.local` next to this notebook (AWS profile, region, model choice, `LIVE` switch) and prints those settings. Credentials come from your AWS SSO profile, never from the notebook or the file. With `LIVE` unset or `0`, every call returns a mock response: no AWS access, no cost. See `00_credentials.ipynb` to set up.

`FAST`, `SMART` and `ALT` are the three Bedrock models from `.env.local` (defaults: Claude Haiku 5.5, Claude Opus 5.5, Amazon Nova 2 Lite).

In [1]:
from _env import ALT, FAST, LIVE, MODELS, SMART, mock, status  # loads .env.local; import before litellm

import litellm

litellm.suppress_debug_info = True
status()

env file    : .env.local (found)
mode        : LIVE (real Bedrock calls, real cost)
AWS_PROFILE : set (from .env.local)
AWS_REGION  : us-west-2
MODEL_FAST  : bedrock/us.anthropic.claude-haiku-5-5 (default)
MODEL_SMART : bedrock/us.anthropic.claude-opus-5-5 (default)
MODEL_ALT   : bedrock/us.amazon.nova-2-lite-v1:0 (default)


## 1. One call, any model

The `bedrock/` prefix picks the Bedrock adapter; the rest is the Bedrock model or inference-profile ID. Claude and Amazon Nova have different native request formats on Bedrock, and the same OpenAI-shaped call works for both. [Section 1](../TUTORIAL.md#1-one-call-any-provider)

`max_tokens` is generous throughout: Claude 5.5 models think by default, and thinking tokens count against `max_tokens`. A tight limit can end the reply before any text.

In [2]:
from litellm import completion, completion_cost

messages = [{"role": "user", "content": "Say hello in French, in three words."}]

for model in MODELS:
    resp = completion(model=model, messages=messages, max_tokens=1024,
                      **mock("Bonjour à tous !"))
    print(f"{model:45} -> {resp.choices[0].message.content!r}")
    print(f"{'':45}    tokens in/out={resp.usage.prompt_tokens}/"
          f"{resp.usage.completion_tokens}  cost=${completion_cost(resp):.8f}")

bedrock/us.anthropic.claude-haiku-5-5         -> "Bonjour, comment allez-vous?\n\nWait, that's more than three words. Here's one that fits:\n\nBonjour tout le monde."
                                                 tokens in/out=19/48  cost=$0.00002849


bedrock/us.anthropic.claude-opus-5-5          -> '**Bonjour, mon ami.**'
                                                 tokens in/out=19/64  cost=$0.00149160


bedrock/us.amazon.nova-2-lite-v1:0            -> '**Bonjour !**  \n\n(Note: "Bonjour !" is actually one word, but if you strictly need three words, you could say: **Salut, bonjour, salut !** – though this is unusual. The traditional and natural way to say “hello” in French is simply **“Bonjour”** or the informal **“Salut”**.)'
                                                 tokens in/out=55/78  cost=$0.00023265


## 2. Streaming

Print text as it arrives, then rebuild the full response with usage. [Section 2](../TUTORIAL.md#2-streaming)

In [3]:
messages = [{"role": "user", "content": "Write a haiku about the sea."}]
stream = litellm.completion(
    model=FAST, messages=messages, stream=True, max_tokens=1024,
    stream_options={"include_usage": True},
    **mock("Grey waves fold and break\nsalt wind carries gull voices\nthe tide keeps its time"),
)

chunks = []
for chunk in stream:
    chunks.append(chunk)
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="|", flush=True)
print(f"\n\nchunks received: {len(chunks)}")

full = litellm.stream_chunk_builder(chunks, messages=messages)
print("rebuilt text:", repr(full.choices[0].message.content))
print("usage:", full.usage.prompt_tokens, "in /", full.usage.completion_tokens, "out")

Waves f|

old into|

 fo|

am,
sal|

t w|

ind car|

... (5 outputs left out)

ide's pat|

ient pull|

.|



chunks received: 16
rebuilt text: "Waves fold into foam,\nsalt wind carries gull cries past\nthe tide's patient pull."
usage: 19 in / 33 out


## 3. Async and concurrent batches

Jupyter already runs an event loop, so use top-level `await` instead of `asyncio.run()`. [Section 3](../TUTORIAL.md#3-async-and-concurrent-batches)

In [4]:
import asyncio
import time

REVIEWS = [
    "Arrived broken, support never answered.",
    "Exactly as described, fast shipping!",
    "It works, I guess. Nothing special.",
    "Best purchase this year.",
    "Stopped working after two days.",
    "Decent value for the price.",
]
FAKE = ["negative", "positive", "neutral", "positive", "negative", "neutral"]
MAX_CONCURRENCY = 3


async def classify(sem, review, fake):
    async with sem:
        resp = await litellm.acompletion(
            model=FAST,
            messages=[
                {"role": "system", "content": "Reply with one word: positive, negative or neutral."},
                {"role": "user", "content": review},
            ],
            max_tokens=1024,
            **mock(fake), mock_delay=None if LIVE else 1.0,
        )
        return (resp.choices[0].message.content or "").strip()


sem = asyncio.Semaphore(MAX_CONCURRENCY)
start = time.perf_counter()
labels = await asyncio.gather(*(classify(sem, r, f) for r, f in zip(REVIEWS, FAKE, strict=True)))
elapsed = time.perf_counter() - start
for review, label in zip(REVIEWS, labels, strict=True):
    print(f"{label:9} {review}")
print(f"\n{len(REVIEWS)} calls, {MAX_CONCURRENCY} at a time: {elapsed:.1f}s")

Negative  Arrived broken, support never answered.
Positive  Exactly as described, fast shipping!
Neutral   It works, I guess. Nothing special.
Positive  Best purchase this year.
Negative  Stopped working after two days.
Positive  Decent value for the price.

6 calls, 3 at a time: 2.9s
